# 🎓 Student Performance Prediction
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1-b6Jf6io10mEnceh52oc64c1i2DpLHGD?usp=sharing)

> **Google Colab Notebook**: [https://colab.research.google.com/drive/1-b6Jf6io10mEnceh52oc64c1i2DpLHGD?usp=sharing](https://colab.research.google.com/drive/1-b6Jf6io10mEnceh52oc64c1i2DpLHGD?usp=sharing)


In [ ]:
# Install required dependencies in Google Colab
!pip install -q gradio xgboost


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from google.colab import files

# 1. Dataset Loading with automated GitHub fallback
print("Upload your dataset (or skip to automatically load sample data from GitHub):")
try:
    uploaded = files.upload()
except Exception:
    uploaded = {}

if uploaded:
    csv_file = next(iter(uploaded.keys()))
    df = pd.read_csv(csv_file)
    print(f"Successfully loaded uploaded file: {csv_file}")
else:
    raw_url = "https://raw.githubusercontent.com/kishorekumar28114/ML-Project/main/sample_data.csv"
    df = pd.read_csv(raw_url)
    print(f"Using fallback sample dataset from GitHub ({len(df)} records)")

target = "Grade"
df = df.dropna()

# 2. Dynamic Categorical Label Encoding
label_encoders = {}
for col in df.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

feature_names = df.drop(target, axis=1).columns.tolist()
X = df.drop(target, axis=1)
y = df[target]

# 3. Train / Test Split (Split BEFORE scaling to prevent data leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Feature Standardization (Fit on train, transform on test)
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)

# 5. Multi-Model Training & Benchmarking
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "XGBoost": XGBRegressor(eval_metric="rmse", random_state=42)
}

results = {}
preds = {}

for name, m in models.items():
    m.fit(X_train_scaled, y_train)
    p = m.predict(X_test_scaled)
    preds[name] = p
    results[name] = r2_score(y_test, p)

res = pd.DataFrame(results.items(), columns=["Model", "R2"]).sort_values(by="R2", ascending=False)

best = res.iloc[0]["Model"]
best_model = models[best]
best_pred = preds[best]

# Visualizations
plt.figure(figsize=(8, 4))
sns.barplot(x="R2", y="Model", data=res, palette="viridis")
plt.title("Model R² Comparison")
plt.xlabel("R² Score")
plt.tight_layout()
plt.show()

# Feature Importance using the benchmarked Random Forest model
rf = models["Random Forest"]
importances = rf.feature_importances_
imp = pd.DataFrame({"Feature": feature_names, "Importance": importances}).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x="Importance", y="Feature", data=imp, palette="magma")
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Gini Importance")
plt.tight_layout()
plt.show()

print(res.to_string(index=False))
print("\nBest Model:", best)
print(f"Test R2   : {r2_score(y_test, best_pred):.4f}")
print(f"Test MSE  : {mean_squared_error(y_test, best_pred):.4f}")


In [ ]:
import gradio as gr

def predict_grade(*args):
    input_dict = {col: args[i] for i, col in enumerate(feature_names)}
    input_df_row = pd.DataFrame([input_dict])

    # Apply stored label encoders
    for col in feature_names:
        if col in label_encoders:
            input_df_row[col] = label_encoders[col].transform(input_df_row[col])

    # Ensure numeric columns are properly typed
    for col in feature_names:
        if col not in label_encoders:
            input_df_row[col] = pd.to_numeric(input_df_row[col], errors="coerce")

    scaled_input = sc.transform(input_df_row[feature_names])
    prediction = best_model.predict(scaled_input)
    return round(float(prediction[0]), 2)

# Dynamic input controls (dropdowns for categorical, number fields for numeric)
gradio_inputs = []
for col_name in feature_names:
    if col_name in label_encoders:
        choices = label_encoders[col_name].classes_.tolist()
        gradio_inputs.append(gr.Dropdown(choices=choices, label=col_name, value=choices[0]))
    elif np.issubdtype(df[col_name].dtype, np.number):
        median_val = float(df[col_name].median())
        gradio_inputs.append(gr.Number(label=col_name, value=median_val))
    else:
        gradio_inputs.append(gr.Textbox(label=col_name))

iface = gr.Interface(
    fn=predict_grade,
    inputs=gradio_inputs,
    outputs=gr.Number(label="Predicted Grade"),
    title="🎓 Student Performance Prediction",
    description=f"Predict a student's grade in real time using the champion model ({best}).",
    theme="default"
)

iface.launch(share=True, debug=True)
